In [7]:
%load_ext autoreload
%autoreload 2
import helper_functions as hf
from imports import *
import importlib
from pathlib import Path

num_available_cpus = multiprocessing.cpu_count()
print("Number of available CPUs:", num_available_cpus)

torch.cuda.empty_cache()
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Device =", device)
torch.set_default_tensor_type('torch.cuda.FloatTensor') if torch.cuda.is_available() else print ('cpu')

torch.set_num_threads(num_available_cpus)

print("Number of threads:", torch.get_num_threads())
print("Number of interop threads:", torch.get_num_interop_threads())

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Number of available CPUs: 80
Device = cuda:0
Number of threads: 80
Number of interop threads: 80


# Evaluating against trainings on dataSB (2017)

In [8]:
year = 2017
signal_samples = ["Qstar2000_W400_UL17","Wp3000_B400_UL17","XYY_X3000_Y80_UL17"]

bkg_flowName = "dataSB_{0}_clip10_NSRATQUAD_k6_hf120_nbpl4_tb10_pt300.pt".format(year)

sig_flowNames = [sig+"_dataSB_{0}_clip10_NSRATQUAD_k6_hf120_nbpl4_tb10_pt300.pt".format(year) for sig in signal_samples]

bkg_flow = hf.load_model(name=bkg_flowName)
sig_flows = [hf.load_model(name=signame) for signame in sig_flowNames]

# Make paths for saving output
Path("evaluations/dataSB/{0}/nominal".format(year)).mkdir(parents=True,exist_ok=True)

In [9]:
# Evaluate on QCD background, normalized using dataSB mean/std
with h5py.File("bkgMeanStds/dataSB_{0}.h5".format(year)) as f:
    mean = f["mean"][()]
    std = f["std"][()]
for i in range(36):
    norm, unnorm, mass = hf.load_bkg_batch_unnorm("QCDBKG",year,i,Mjj_cut=800,pt_cut=300)
    norm = hf.normalize_data(norm,inp_mean=mean,inp_std=std)
    bkg_loss = -bkg_flow.eval_log_prob(norm)[0]
    sig_losses = [-sflow.eval_log_prob(norm)[0] for sflow in sig_flows]
    with h5py.File("evaluations/dataSB/{0}/nominal/eval_QCDBKG_{1}.h5".format(year,i),"w") as f:
        f.create_dataset("mass",data=mass[:,0])
        f.create_dataset("dataSB_{0}".format(year),data=bkg_loss)
        for j in range(len(sig_losses)):
            f.create_dataset(signal_samples[j],data=sig_losses[j])
    del norm, unnorm, mass, bkg_loss, sig_losses

In [10]:
# Evaluate on signals, normalized using dataSB mean/std
with h5py.File("bkgMeanStds/dataSB_{0}.h5".format(year)) as f:
    mean = f["mean"][()]
    std = f["std"][()]
for sig_samp in signal_samples:
    norm, unnorm, mass, varnames = hf.LAPS_train(sig_samp,year,num_batches=1,Mjj_cut=800,pt_cut=300)
    norm = hf.normalize_data(norm,inp_mean=mean,inp_std=std)
    bkg_loss = -bkg_flow.eval_log_prob(norm)[0]
    sig_losses = [-sflow.eval_log_prob(norm)[0] for sflow in sig_flows]
    with h5py.File("evaluations/dataSB/{0}/nominal/eval_{1}.h5".format(year,sig_samp),"w") as f:
        f.create_dataset("mass",data=mass[:,0])
        f.create_dataset("dataSB_{0}".format(year),data=bkg_loss)
        for j in range(len(sig_losses)):
            f.create_dataset(signal_samples[j],data=sig_losses[j])
    del norm, unnorm, mass, bkg_loss, sig_losses

# Evaluate against trainings on QCDBKG MC (2017)

In [8]:
year = 2017
signal_samples = ["Qstar2000_W400_UL17","Wp3000_B400_UL17","XYY_X3000_Y80_UL17"]

bkg_names = ["QCDBKG_{0}".format(year),"qcd_{0}".format(year),"top_{0}".format(year),"vjets_{0}".format(year)]
bkg_flowNames = [name+"_clip10_NSRATQUAD_k6_hf120_nbpl4_tb10_pt300.pt" for name in bkg_names]
bkg_flows = [hf.load_model(name=bkgname) for bkgname in bkg_flowNames]

sig_flowNames = [sig+"_QCDBKG_{0}_clip10_NSRATQUAD_k6_hf120_nbpl4_tb10_pt300.pt".format(year) for sig in signal_samples]
sig_flows = [hf.load_model(name=signame) for signame in sig_flowNames]

# Make paths for saving output
Path("evaluations/qcdbkg/{0}/nominal".format(year)).mkdir(parents=True,exist_ok=True)

In [9]:
# Evaluate on QCD background, normalized using QCDBKG mean/std
with h5py.File("bkgMeanStds/QCDBKG_{0}.h5".format(year)) as f:
    mean = f["mean"][()]
    std = f["std"][()]
for i in range(36):
    norm, unnorm, mass = hf.load_bkg_batch_unnorm("QCDBKG",year,i,Mjj_cut=800,pt_cut=300)
    norm = hf.normalize_data(norm,inp_mean=mean,inp_std=std)
    bkg_losses = [-bflow.eval_log_prob(norm)[0] for bflow in bkg_flows]
    sig_losses = [-sflow.eval_log_prob(norm)[0] for sflow in sig_flows]
    with h5py.File("evaluations/qcdbkg/{0}/nominal/eval_QCDBKG_{1}.h5".format(year,i),"w") as f:
        f.create_dataset("mass",data=mass[:,0])
        for j in range(len(bkg_losses)):
            f.create_dataset(bkg_names[j],data=bkg_losses[j])
        for j in range(len(sig_losses)):
            f.create_dataset(signal_samples[j],data=sig_losses[j])
    del norm, unnorm, mass, bkg_losses, sig_losses

In [10]:
# Evaluate on signals, normalized using QCDBKG mean/std
with h5py.File("bkgMeanStds/QCDBKG_{0}.h5".format(year)) as f:
    mean = f["mean"][()]
    std = f["std"][()]
for sig_samp in signal_samples:
    norm, unnorm, mass, varnames = hf.LAPS_train(sig_samp,year,num_batches=1,Mjj_cut=800,pt_cut=300)
    norm = hf.normalize_data(norm,inp_mean=mean,inp_std=std)
    bkg_losses = [-bflow.eval_log_prob(norm)[0] for bflow in bkg_flows]
    sig_losses = [-sflow.eval_log_prob(norm)[0] for sflow in sig_flows]
    with h5py.File("evaluations/qcdbkg/{0}/nominal/eval_{1}.h5".format(year,sig_samp),"w") as f:
        f.create_dataset("mass",data=mass[:,0])
        for j in range(len(bkg_losses)):
            f.create_dataset(bkg_names[j],data=bkg_losses[j])
        for j in range(len(sig_losses)):
            f.create_dataset(signal_samples[j],data=sig_losses[j])
    del norm, unnorm, mass, bkg_losses, sig_losses